In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

RANDOM_STATE = 42
MODELS_DIR   = '../data/models/predictor'
SPLIT_DIR    = '../data/splits'

Librerías cargadas correctamente.


In [2]:
X_train = pd.read_csv(f'{SPLIT_DIR}/X_train.csv')
y_train = pd.read_csv(f'{SPLIT_DIR}/y_train.csv').squeeze()

print(f'\nDistribución de is_open:')
print(y_train.value_counts())

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'\nscale_pos_weight: {scale_pos_weight:.4f}')

Train: 120,194 instancias, 63 features

Distribución en train:
is_open
1    95682
0    24512
Name: count, dtype: int64

scale_pos_weight: 0.2562


In [3]:
pipeline_xgb = Pipeline([
    ('clf', XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    ))
])

param_grid = {
    'clf__n_estimators':  [100, 200, 300],
    'clf__max_depth':     [4, 6, 8],
    'clf__learning_rate': [0.05, 0.1, 0.2],
    'clf__subsample':     [0.7, 0.8, 0.9],
}

n_combinaciones = 1
for v in param_grid.values():
    n_combinaciones *= len(v)

print(f'Combinaciones a evaluar: {n_combinaciones}')
print(f'Entrenamientos totales (5-fold CV): {n_combinaciones * 5}')

Combinaciones a evaluar: 81
Entrenamientos totales (5-fold CV): 405
Tiempo estimado: 10.1 min aprox.


In [4]:
grid_search = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True
)

t0 = time.time()
grid_search.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'\nCompletado en {elapsed / 60:.1f} min')

Iniciando GridSearchCV...
Fitting 5 folds for each of 81 candidates, totalling 405 fits

Completado en 4.8 min


In [5]:
print('Mejores hiperparámetros encontrados:')
for param, valor in grid_search.best_params_.items():
    print(f'  {param}: {valor}')

print(f'\nMejor ROC-AUC en validación cruzada: {grid_search.best_score_:.4f}')

Mejores hiperparámetros encontrados:
  clf__learning_rate: 0.05
  clf__max_depth: 6
  clf__n_estimators: 300
  clf__subsample: 0.8

Mejor ROC-AUC en validación cruzada: 0.7925
ROC-AUC del modelo base (sin GridSearch): 0.7926


In [6]:
resultados_cv = pd.DataFrame(grid_search.cv_results_)
top10 = (
    resultados_cv[[
        'param_clf__n_estimators',
        'param_clf__max_depth',
        'param_clf__learning_rate',
        'param_clf__subsample',
        'mean_test_score',
        'std_test_score',
        'rank_test_score'
    ]]
    .sort_values('rank_test_score')
    .head(10)
    .rename(columns={
        'param_clf__n_estimators':  'n_estimators',
        'param_clf__max_depth':     'max_depth',
        'param_clf__learning_rate': 'learning_rate',
        'param_clf__subsample':     'subsample',
        'mean_test_score':          'ROC-AUC (mean)',
        'std_test_score':           'ROC-AUC (std)',
        'rank_test_score':          'rank'
    })
)

print('Top 10 combinaciones por ROC-AUC en validación cruzada:')
print(top10.to_string(index=False))

Top 10 combinaciones por ROC-AUC en validación cruzada:
 n_estimators  max_depth  learning_rate  subsample  ROC-AUC (mean)  ROC-AUC (std)  rank
          300          6           0.05        0.8        0.792522       0.003054     1
          300          6           0.05        0.7        0.792290       0.003314     2
          300          6           0.05        0.9        0.792062       0.003591     3
          200          8           0.05        0.7        0.792020       0.003704     4
          200          6           0.10        0.8        0.791827       0.003041     5
          300          4           0.10        0.8        0.791521       0.003447     6
          200          8           0.05        0.8        0.791509       0.003805     7
          300          4           0.10        0.9        0.791463       0.003458     8
          200          6           0.10        0.7        0.791441       0.003578     9
          200          6           0.10        0.9        0.7913

In [ ]:
output_path = f'{MODELS_DIR}/xgboost_optimized.pkl'
joblib.dump(grid_search.best_estimator_, output_path)

size = os.path.getsize(output_path) / 1024 / 1024
print(f'Modelo guardado en: {output_path}')